# Phase 4: The agent

Needs `OPENROUTER_API_KEY` in `../.env`. Each `ask` calls Claude Opus 5.5 through OpenRouter (`anthropic/claude-opus-5.5`).

In [2]:
import sys
sys.path.insert(0, '..')

from welllog.agent import WellLogAgent
from welllog.tools import curve_stats

agent = WellLogAgent('../data/sample.las')

### 1. Lists the real curves

In [3]:
print(agent.ask('What curves are in this file?'))

The Busse #1-27 file has **22 curves**. The log runs from 345 to 5209 ft with a 0.5 ft sample step.

**Gamma ray / SP**
- **GR**: Gamma Ray (GAPI), valid 345–5167 ft
- **SP**: DIL Spontaneous Potential (MV), valid 345–5209 ft

**Resistivity / conductivity (Dual Induction)**
- **RILD**: Deep Resistivity (OHM-M), valid 345–5209 ft
- **RILM**: Medium Resistivity (OHM-M), valid 345–5209 ft
- **RLL3**: Shallow Resistivity (OHM-M), valid 345–5195.5 ft
- **CILD**: Deep Conductivity (MMHO/M), valid 345–5209 ft
- **RXORT**: Rxo/Rt ratio (no unit), valid 345–5209 ft

**Microlog**
- **MN**: Micro Normal Resistivity (OHM-M), valid 345–5192 ft
- **MI**: Micro Inverse Resistivity (OHM-M), valid 345–5192 ft
- **MCAL**: Microlog caliper (IN), valid 345–5192.5 ft

**Porosity: neutron**
- **CNLS**: Neutron porosity, limestone matrix (PU), valid 345–5169 ft
- **CNSS**: Neutron porosity, sandstone matrix (PU), valid 345–5169 ft
- **CNDL**: Neutron porosity, dolomite matrix (PU), valid 345–5169 ft

**Poros

### 2. Porosity matches a direct `curve_stats` call

This well is in **feet**, so the question is asked in ft. (Asking in m is check 4.)

In [4]:
print(agent.ask('Average porosity between 2500 and 2700 ft?'))
print('\nDirect:', {k: curve_stats(agent.df, 'CNLS', 2500, 2700)[k] for k in ('curve', 'unit', 'count', 'mean')})

The average porosity from 2500 to 2700 ft is about **29 PU** on both porosity logs (401 valid samples each, every 0.5 ft):

| Curve | Mean | P50 | P10–P90 | Min–Max |
|---|---|---|---|---|
| **CNLS** (neutron porosity, limestone scale) | 28.7 PU | 27.2 | 21.3–38.8 | 12.1–49.9 |
| **DPOR** (density porosity) | 29.1 PU | 27.7 | 19.6–39.5 | 11.8–69.1 |

**Which curve:** The well has several porosity logs. I used neutron porosity on the limestone scale and density porosity. Neutron porosity is also available on sandstone (CNSS) and dolomite (CNDL) scales; those values will differ if the rock isn't limestone.

**Caveats:**
- **The highest density reading may be bad data.** The DPOR maximum of 69.1 PU is unrealistically high. It's likely caused by an enlarged or washed-out borehole. I didn't check the caliper (DCAL), so it could be pulling the DPOR mean up slightly.
- **These may not be true porosity.** Neutron and density readings this high (~29 PU) can be partly caused by shale. Checking t

### 3. Flags high-GR zones and plots them

In [7]:
print(agent.ask('Flag zones with high gamma ray and plot them with porosity.'))
agent.figures[-1].show() if agent.figures else None

I flagged **65 high gamma ray zones**, **398.5 ft** thick in total, and plotted them over the porosity curves.

**How I set it up**
- **Gamma ray curve:** GR (GAPI). It has valid data from 345 to 5167 ft.
- **What counts as "high":** GR above **125.1 GAPI**. That's the value 90% of the well's GR readings fall below (P90). The median is 86.2 GAPI, so this picks out the most radioactive intervals in this well rather than relying on a fixed shale cutoff.
- **Minimum thickness:** 2 ft, my choice, to filter out single-sample spikes.
- **Porosity curves:** the well has several, so I plotted CNLS (neutron porosity, limestone matrix, PU) and DPOR (density porosity, PU). The neutron curve also comes in sandstone and dolomite versions (CNSS, CNDL) if your lithology calls for one of those.

**Plot:** three tracks (GR, CNLS, DPOR) over the full logged interval, 345–5209 ft, with the high-GR zones shaded.

**Main zones** (the tool listed only the first 50 of the 65, down to 4594 ft):
| Interval (ft

### 4. Bad input gives a helpful reply

In [6]:
print(agent.ask('What is the average NPHI?'))
print('---')
print(agent.ask('Average porosity between 2500 and 2700 m?'))
print('---')
print(agent.ask('Average gamma ray between 8000 and 9000 ft?'))

This well has no curve called NPHI. The neutron porosity is recorded as three curves, one for each matrix: **CNLS** (limestone), **CNSS** (sandstone) and **CNDL** (dolomite). I used **CNLS** because limestone-matrix neutron is the usual default for "NPHI." If you want the sandstone or dolomite version, I can use that instead.

**CNLS (CN Limestone Porosity) over 345–5209 ft:**
- **Mean: 26.1 PU**
- Median (P50): 27.2 PU; P10–P90: 7.5–42.1 PU
- Range: −1.126 to 58.85 PU
- 9,649 valid samples. The curve has data down to 5169 ft, so the bottom 40 ft of the log has no neutron readings.

The data needs some care in two places:
- **Negative values:** the minimum of −1.126 PU is slightly negative, which can't be real porosity. It's probably a matrix-correction effect in very tight or anhydritic beds.
- **Shale bias:** this is a whole-well average that includes shales, which push neutron porosity up. It won't represent reservoir porosity. For that, I can work out the average over specific zone